In [ ]:
# ============================================================================
# PRODUCTION SCRIPT: STANDALONE YOLO TO JSON EXPORT PIPELINE
# ============================================================================
import os
import json
from pathlib import Path
from datetime import datetime
from collections import Counter
from ultralytics import YOLO

class MilkYOLOPredictor:

    def __init__(
        self,
        weights_path="/workspace/shared/food-guard-ai/notebooks/runs/detect/milk_detection_framework/yolo12_classes_run-3/weights/best.pt",
        output_dir="output/yolo_predictions"
    ):
        """
        Initializes the model, extracts class names, and sets up the output directory.
        """
        print("🚀 Loading YOLO model architecture...")
        self.model = YOLO(weights_path)

        # Pull and sort all 12 target categories alphabetically from your model config
        self.all_classes = sorted([v for k, v in self.model.names.items()])

        # Establish the target JSON file folder location
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        print("🏷️ Initialized Master 12-Class Registry:")
        print(self.all_classes)
        print(f"📂 JSON export directory ready at: {self.output_dir.absolute()}\n")

    def predict_folder(self, image_folder, conf=0.25):
        """
        Loops through an input directory to process all valid image frames.
        """
        image_folder = Path(image_folder)
        if not image_folder.exists():
            raise FileNotFoundError(f"Target folder not found: {image_folder}")

        valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
        total = 0

        for image_file in image_folder.iterdir():
            if image_file.suffix.lower() not in valid_ext:
                continue

            self.predict_image(image_path=str(image_file), conf=conf)
            total += 1

        print(f"\n✓ Complete! Exported {total} JSON result files to disk.")

    def predict_image(self, image_path, conf=0.25):
        """
        Runs prediction on a single image file and builds the strict 12-class JSON output.
        """
        image_file = Path(image_path)
        print(f"🔍 Analyzing Frame: {image_file.name}")

        # Run inference using standard CPU execution for system stability
        results = self.model.predict(source=str(image_file), conf=conf, device='cpu', verbose=False)
        result = results[0]

        detected_labels = []
        max_conf = 0.0

        # Process bounding boxes ONLY if the model actually detects them
        if len(result.boxes) > 0:
            max_conf = float(result.boxes.conf.max().item())
            for box in result.boxes:
                cls_id = int(box.cls[0].item())
                detected_labels.append(self.model.names[cls_id])
        else:
            max_conf = 0.0

        # Tally occurrences from our detection bucket
        found_counts = Counter(detected_labels)

        # Build the exact 12-class dictionary payload (unseen classes auto-default to 0)
        full_12_class_counts = {cname: found_counts.get(cname, 0) for cname in self.all_classes}

        # Structure metadata tokens according to your swapped mapping requirements
        sample_id = image_file.name                  # <-- data_sample_id is now the filename
        absolute_image_path = str(image_file.resolve()) # <-- image_filename is now the absolute path

        total_objects = len(detected_labels)
        unique_classes_list = sorted(list(set(detected_labels)))
        timestamp_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Assemble the finalized structured dataset record
        payload = {
            "data_sample_id": sample_id,
            "image_filename": absolute_image_path,
            "class_counts_json": full_12_class_counts,  # True native dense dictionary mapping
            "total_objects": total_objects,
            "unique_classes": unique_classes_list,      # Clean JSON array tracking
            "created_at": timestamp_now
        }

        # Save to file using the image filename as the base tracking name
        self.save_json(payload, filename=image_file.name)

        print(f"   ↳ ID (Name)   : {sample_id}")
        print(f"   ↳ File Path   : {absolute_image_path}")
        print(f"   ↳ Objects     : {total_objects} | Score: {round(max_conf, 4)} | Classes: {unique_classes_list}")

    def save_json(self, payload, filename):
        """
        Writes the prediction payload map directly to an individual file on disk.
        """
        # Saves as e.g., output/yolo_predictions/detergent_added_0016.jpg.json
        json_path = self.output_dir / f"{filename}.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=4)  # Generates clean, human-readable JSON formats


In [ ]:
if __name__ == "__main__":

    # Define paths according to your workspace environment layout


    # Initialize the engine
    predictor = MilkYOLOPredictor( weights_path=MODEL_WEIGHTS,
        output_dir=JSON_OUTPUT_FOLDER
    )

    # Execute batch folder processing loop safely within the catch architecture
    try:
        predictor.predict_folder(image_folder=INPUT_VAL_FOLDER, conf=0.25)
    except Exception as e:
        print(f"❌ Execution stopped with an operational error: {e}")